In [ ]:
import torch
from OmniTokenizer import OmniTokenizer

# Loading the Tokenizer

In [ ]:
# load model
# weights are saved in bfloat16, so be sure to convert
model = torch.load("omnibiota-multi-E-mixed-small.pt", map_location="cpu").bfloat16().eval()
model_single_char = torch.load("omnibiota-multi-E-mixed-small-char.pt", map_location="cpu").bfloat16().eval()

# load sentencepiece tokenizer
tokenizer = OmniTokenizer(tokenizer_dir=".", single_char=False) # tokenizer_dir should contain genbank-2k.model and uniref-2k.model
tokenizer_single_char = OmniTokenizer(single_char=True) # for single-char mode, no tokenizer dir is needed

# Generating Embeddings

In [ ]:
# valid header tokens are <DNA> <mRNA> <RNA> <rRNA> <tRNA> <cRNA> <ss-RNA> <ss-DNA> <ds-mRNA> <ds-rRNA> <ds-RNA> <ms-DNA> <ms-RNA> <ds-cRNA>
# keep in mind that some of the more infrequent nucleic acid types may behave unexpectedly as they had very little training data
# for a table of the amount of training data for each sequence, see the supplementary information in our pre-print (https://arxiv.org/abs/2408.16245)
nuc_seq = "<DNA>ACGTAGATCGATCGCACTAGTAGTCAGCATGCA<EOS>"

# the only valid header for protein sequences is <protein>
prot_seq = "<protein>ACDEFGHIKLMNPQRSTVWYACDEFGHIKLMNPQRSTVWY<EOS>"

with torch.no_grad():
    nuc_tokens = tokenizer.Encode(nuc_seq, seq_type="nuc") # seq_type must be specified
    prot_tokens = tokenizer.Encode(prot_seq, seq_type="prot") # seq_type must be specified

    input_seq = torch.tensor(nuc_tokens + prot_tokens, dtype=torch.long).unsqueeze(0) # prepare input

    # forward pass through the model to get embeddings
    emb = model(input_seq, return_embeddings=True)

# analogous code for the single-char model
with torch.no_grad():
    nuc_tokens = tokenizer_single_char.Encode(nuc_seq, seq_type="nuc") # seq_type must be specified
    prot_tokens = tokenizer_single_char.Encode(prot_seq, seq_type="prot") # seq_type must be specified

    input_seq = torch.tensor(nuc_tokens + prot_tokens, dtype=torch.long).unsqueeze(0) # prepare input

    # forward pass through the model to get embeddings
    emb_single_char = model_single_char(input_seq, return_embeddings=True)

# $\Delta G$ Prediction

In [ ]:
# first, we load the model and tokenizer
# the tokenizer in this case is the sentencepiece tokenizer

tokenizer = OmniTokenizer(tokenizer_dir=".", single_char=False) # tokenizer_dir should contain genbank-2k.model and uniref-2k.model
model = torch.load("omnibinder-XL.pt", map_location="cpu").bfloat16().eval()

# p53 (homo sapien) 
prot_seq = "<protein>MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHERCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELPPGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPGGSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD<EOS>"

# associated motif
# for double stranded DNA, we append the other strand of the sequence in the 5'-->3'
# the other strand is often the complement, as in this case
nuc_seq = "<DNA>GGGCATGCCCGGGCATGCCC<EOS>GGGCATGCCCGGGCATGCCC<EOS>"

delta_g = model(torch.tensor(tokenizer.Encode(prot_seq, seq_type="prot") + tokenizer.Encode(nuc_seq, seq_type="nuc"), dtype=torch.long).unsqueeze(0)).item()
print(delta_g) # ground truth is -10.20

# Serine/arginine-rich splicing factor 5
prot_seq = "<protein>MSGCRVFIGRLNPAAREKDVERFFKGYGRIRDIDLKRGFGFVEFEDPRDADDAVYELDGKELCSERVTIEHARARSRGGRGRGRYSDRFSSRRPRNDRRNAPPVRTENRLIVENLSSRVSWQDLKDFMRQAGEVTFADAHRPKLNEGVVEFASYGDLKNAIEKLSGKEINGRKIKLIEGSKRHSRSRSRSRSRTRSSSRSRSRSRSRSRKSYSRSRSRSRSRSRSKSRSVSRSPVPEKSQKRGSSSRSKSPASVDRQRSRSRSRSRSVDSGN<EOS>"

nuc_seq = "<RNA>TGAAGGAC<EOS>" # for RNA, we convert "U" to "T" as convention

delta_g = model(torch.tensor(tokenizer.Encode(prot_seq, seq_type="prot") + tokenizer.Encode(nuc_seq, seq_type="nuc"), dtype=torch.long).unsqueeze(0)).item()
print(delta_g) # ground truth is -6.78